In [ ]:
!pip install datasets
!pip install -U sentence-transformers
!sudo apt-get install git-lfs

     |████████████████████████████████| 311 kB 5.6 MB/s 
     |████████████████████████████████| 243 kB 11.0 MB/s 
     |████████████████████████████████| 67 kB 3.7 MB/s 
     |████████████████████████████████| 1.1 MB 40.8 MB/s 
     |████████████████████████████████| 133 kB 33.9 MB/s 
     |████████████████████████████████| 144 kB 42.7 MB/s 
     |████████████████████████████████| 271 kB 20.5 MB/s 
     |████████████████████████████████| 94 kB 426 kB/s 
     |████████████████████████████████| 79 kB 3.6 MB/s 
     |████████████████████████████████| 3.5 MB 11.8 MB/s 
     |████████████████████████████████| 1.2 MB 51.5 MB/s 
     |████████████████████████████████| 596 kB 50.8 MB/s 
     |████████████████████████████████| 6.8 MB 38.7 MB/s 
     |████████████████████████████████| 895 kB 56.1 MB/s 
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.0-py3-none-any.whl size=120747 sha256=100c5e53e85d54b53ad84380f14059543d088f3f2063c96b733946826ea13dd2
  Stored in di

In [ ]:
import json
import math
import pandas as pd
from datasets import load_dataset,Dataset,DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, mean_absolute_percentage_error, r2_score, roc_curve
from sentence_transformers import SentenceTransformer, InputExample, losses, util, evaluation, models
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader
from torch import nn

In [ ]:
import datasets
import sklearn
import sentence_transformers
import torch

In [ ]:
!git config --global user.email "Enter your email here"
!git config --global user.name "Enter your name here"

# Need to login to Hugging Face to push the model to the hub. You can create an account at https://huggingface.co/join and get your token from https://huggingface.co/settings/tokens
!huggingface-cli login 


        _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
        _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
        _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
        _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
        _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

        To login, `huggingface_hub` now requires a token generated from https://huggingface.co/settings/token.
        (Deprecated, will be removed in v0.3.0) To login with username and password instead, interrupt with Ctrl+C.
        
Token: 
Login successful
Your token has been saved to /root/.huggingface/token
Authenticated through git-credential store but this isn't the helper defined on your machine

In [ ]:
def ConvertJsonToList(fileName, split):
  subject1_fileDataset = load_dataset("json", data_files=fileName, split="train")
  samples = []

  for i in range (0,len(subject1_fileDataset[split][0])): #len(subject1)
    mark = subject1_fileDataset[split][0][i]['mark']
    responseStudent = subject1_fileDataset[split][0][i]['responseStudent']
    responseTeacher = subject1_fileDataset[split][0][i]['responseTeacher']
    
    ie = InputExample(texts=[responseTeacher, responseStudent], label=mark)
    samples.append(ie)

  return samples    

In [ ]:
#Creating a list with the necesarry fields
data_train = ConvertJsonToList("JoinedSubjects.json", "train")
data_test = ConvertJsonToList("JoinedSubjects.json", "test")
data_valid = ConvertJsonToList("JoinedSubjects.json", "valid")

Using custom data configuration default-59fe525fd3bf0d42


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Dataset json downloaded and prepared to /root/.cache/huggingface/datasets/json/default-59fe525fd3bf0d42/0.0.0/ac0ca5f5289a6cf108e706efcf040422dbbfa8e658dee6a819f20d76bb84d26b. Subsequent calls will reuse this data.


Using custom data configuration default-59fe525fd3bf0d42
Reusing dataset json (/root/.cache/huggingface/datasets/json/default-59fe525fd3bf0d42/0.0.0/ac0ca5f5289a6cf108e706efcf040422dbbfa8e658dee6a819f20d76bb84d26b)
Using custom data configuration default-59fe525fd3bf0d42
Reusing dataset json (/root/.cache/huggingface/datasets/json/default-59fe525fd3bf0d42/0.0.0/ac0ca5f5289a6cf108e706efcf040422dbbfa8e658dee6a819f20d76bb84d26b)


In [ ]:
def CreateModelEvaluationData():
  sentences1 = []
  sentences2 = []
  scores = []

  for i in range (0,len(data_valid)):
    sentences1.append(data_valid[i].texts[0])
    sentences2.append(data_valid[i].texts[1])
    scores.append(data_valid[i].label)

  evaluator = evaluation.EmbeddingSimilarityEvaluator(sentences1, sentences2, scores)
  return evaluator

In [ ]:
def TrainModel(checkpoint, evaluator, epochs, fromScratch):
  batch_size = int(len(data_train) * 0.1)
  #Create the model from checkpoint
  if (not fromScratch):
    model = SentenceTransformer(checkpoint)
  else:
    word_embedding_model = models.Transformer(checkpoint, max_seq_length=256)
    pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
    dense_model = models.Dense(in_features=pooling_model.get_sentence_embedding_dimension(), out_features=256, activation_function=nn.Tanh())
    model = SentenceTransformer(modules=[word_embedding_model, pooling_model, dense_model])

  train_dataloader = DataLoader(data_train, shuffle=True, batch_size=batch_size)
  train_loss = losses.CosineSimilarityLoss(model)

  #Fit the model
  local_model_path = 'Model_' + checkpoint + '/' + str(epochs) + '_Epochs'
  warmup_steps = math.ceil(len(train_dataloader) * epochs * 0.1) #10% of train data for warm-up
  evaluation_steps = int(len(train_dataloader)*0.1)
  print(len(train_dataloader),warmup_steps,evaluation_steps)
  model.fit(train_objectives=[(train_dataloader, train_loss)]
            , epochs=epochs
            , warmup_steps=warmup_steps
            , evaluator=evaluator
            , evaluation_steps=evaluation_steps
            ,output_path=local_model_path
            ,save_best_model=True)
  
  repo_name = "jfarray/" + local_model_path.replace('/','_')
  model.save_to_hub(repo_name = repo_name, local_model_path = local_model_path, exist_ok= True, replace_model_card= True)

In [ ]:
modelsToTrain = [
                  {"checkPoint": "distiluse-base-multilingual-cased-v1", "fromScratch": False}
                  {"checkPoint": "paraphrase-multilingual-MiniLM-L12-v2", "fromScratch": False}
                  {"checkPoint": "paraphrase-multilingual-mpnet-base-v2", "fromScratch": False}
                  {"checkPoint": "all-distilroberta-v1", "fromScratch": False}
                  {"checkPoint": "bert-base-multilingual-uncased", "fromScratch": True}
                  {"checkPoint": "dccuchile/bert-base-spanish-wwm-uncased", "fromScratch": True}
              ]

epochsToTest = [1,5,10,30,50,100]

#Get evaluator
evaluator = CreateModelEvaluationData()

#Train the models
for model in modelsToTrain:
  for epochs in epochsToTest:
    TrainModel(model["checkPoint"], evaluator, epochs, model["fromScratch"])

Some weights of the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased were not used when initializing BertModel: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['bert.pooler.dens

11 6 1


/usr/local/lib/python3.7/dist-packages/transformers/optimization.py:309: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use thePyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  FutureWarning,


Epoch:   0%|          | 0/5 [00:00<?, ?it/s]

Iteration:   0%|          | 0/11 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/11 [00:00<?, ?it/s]

/usr/local/lib/python3.7/dist-packages/huggingface_hub/hf_api.py:1004: FutureWarning: `create_repo` now takes `token` as an optional positional argument. Be sure to adapt your code!
  FutureWarning,
Cloning https://huggingface.co/jfarray/Model_dccuchile_bert-base-spanish-wwm-uncased_100_Epochs into local empty directory.


Upload file pytorch_model.bin:   0%|          | 3.35k/419M [00:00<?, ?B/s]

Upload file 2_Dense/pytorch_model.bin:   0%|          | 3.35k/770k [00:00<?, ?B/s]

To https://huggingface.co/jfarray/Model_dccuchile_bert-base-spanish-wwm-uncased_100_Epochs
   2d1a078..d5bd386  main -> main

   2d1a078..d5bd386  main -> main



from google.colab import files
import os

for checkpoint in modelsToTest:
  dir_to_zip = '/content/Model_' + checkpoint 
  output_filename = '/content/'+ checkpoint + '.zip' 
  delete_dir_after_download = "No"  

  os.system( "zip -r {} {}".format( output_filename , dir_to_zip ) )

  if delete_dir_after_download == "Yes":
      os.system( "rm -r {}".format( dir_to_zip ) )

  #files.download( output_filename )